# 01 — Quaternion Intuition

Quaternions look intimidating at first, but they become intuitive once you connect them to geometry.

This notebook builds that intuition step by step — starting from the scalar/vector decomposition, through normalization and conjugation, all the way to the axis-angle picture that links directly to quantum rotations.

---

## What is a quaternion?

A quaternion is a four-component number:

$$q = w + x\mathbf{i} + y\mathbf{j} + z\mathbf{k}$$

| Part | Notation | Meaning |
|---|---|---|
| **Scalar** | $w$ | "how much rotation" information |
| **Vector** | $(x, y, z)$ | encodes the rotation axis |

The imaginary units satisfy $\mathbf{i}^2 = \mathbf{j}^2 = \mathbf{k}^2 = \mathbf{ijk} = -1$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from helpers.notebook_style import setup_notebook
from helpers.plotting import plot_quaternion_components, plot_bloch_like_vector, plot_rotation_path
from helpers.display_utils import show_axis_angle_summary, format_complex_pair
setup_notebook()
print("Setup complete.")


## Scalar and vector parts

For a 90-degree rotation around each principal axis, notice how the scalar ($w$) and vector ($x, y, z$) parts change.

$$q = \cos\!\frac{\theta}{2} + \sin\!\frac{\theta}{2}\,(n_x\mathbf{i} + n_y\mathbf{j} + n_z\mathbf{k})$$

- When $\theta = 0$: $w = 1$, vector = 0 → identity rotation
- When $\theta = \pi$: $w = 0$, vector = axis → 180° rotation


In [ ]:
def unit_quaternion(theta, axis):
    """Unit quaternion for rotation by theta around axis.
    NOTE: use rqm-core in production — this is illustrative.
    """
    n = np.array(axis, dtype=float)
    n /= np.linalg.norm(n)
    w = np.cos(theta / 2)
    xyz = np.sin(theta / 2) * n
    return (w, xyz[0], xyz[1], xyz[2])

# 90-degree rotations around each axis
q_x90 = unit_quaternion(np.pi / 2, [1, 0, 0])
q_y90 = unit_quaternion(np.pi / 2, [0, 1, 0])
q_z90 = unit_quaternion(np.pi / 2, [0, 0, 1])

fig = plot_quaternion_components(
    [q_x90, q_y90, q_z90],
    labels=["Rx(90°)", "Ry(90°)", "Rz(90°)"],
    title="Components of 90° rotations around X, Y, Z",
)
plt.tight_layout()
plt.show()


## Normalization

A quaternion represents a **valid** rotation only if $|q| = 1$:

$$|q| = \sqrt{w^2 + x^2 + y^2 + z^2} = 1$$

Unnormalized quaternions arise from floating-point drift or composition of many rotations.  
`rqm-core` always normalizes internally, but it is good practice to verify.


In [ ]:
# Demonstrate normalization
q_raw = np.array([0.6, 0.2, 0.5, 0.3])  # deliberately not unit
norm_raw = np.linalg.norm(q_raw)
q_norm = q_raw / norm_raw

print(f"Raw quaternion:        {q_raw}")
print(f"|q_raw|               = {norm_raw:.6f}  (not 1)")
print()
print(f"Normalized quaternion: {np.round(q_norm, 4)}")
print(f"|q_normalized|        = {np.linalg.norm(q_norm):.6f}  (exactly 1)")


## Conjugation and inversion

The **conjugate** of $q = w + x\mathbf{i} + y\mathbf{j} + z\mathbf{k}$ is:

$$q^* = w - x\mathbf{i} - y\mathbf{j} - z\mathbf{k}$$

For a unit quaternion, the conjugate equals the inverse: $q^{-1} = q^*$.

Geometrically, the conjugate represents the **same rotation but in the opposite direction**.

$$q \cdot q^* = |q|^2 = 1$$


In [ ]:
def quat_conjugate(q):
    w, x, y, z = q
    return (w, -x, -y, -z)

def quat_mul(q1, q2):
    """Hamilton product (w, x, y, z)."""
    w1, x1, y1, z1 = q1; w2, x2, y2, z2 = q2
    return (
        w1*w2 - x1*x2 - y1*y2 - z1*z2,
        w1*x2 + x1*w2 + y1*z2 - z1*y2,
        w1*y2 - x1*z2 + y1*w2 + z1*x2,
        w1*z2 + x1*y2 - y1*x2 + z1*w2,
    )

q  = unit_quaternion(np.pi / 3, [1, 1, 0])   # 60° around X+Y diagonal
qc = quat_conjugate(q)
qq_star = quat_mul(q, qc)

print(f"q       = ({q[0]:.4f}, {q[1]:.4f}, {q[2]:.4f}, {q[3]:.4f})")
print(f"q*      = ({qc[0]:.4f}, {qc[1]:.4f}, {qc[2]:.4f}, {qc[3]:.4f})")
print()
print("q · q* (should be identity = (1, 0, 0, 0)):")
print(f"  ({qq_star[0]:.6f}, {qq_star[1]:.6f}, {qq_star[2]:.6f}, {qq_star[3]:.6f})")


## Axis-angle representation

Every unit quaternion maps directly to an axis-angle rotation.  
The `show_axis_angle_summary` helper makes this concrete.


In [ ]:
# Hadamard-like rotation: 180° around X+Z diagonal
show_axis_angle_summary(
    axis=[1 / np.sqrt(2), 0, 1 / np.sqrt(2)],
    angle_rad=np.pi,
    label="Hadamard (Rx+z(180°))",
)


## The double cover: q and −q are the same rotation

This is the most important subtlety of quaternions for quantum mechanics:

$$q \text{ and } {-q} \text{ represent the **same physical rotation**}$$

Because we use $\theta/2$ in the quaternion formula, rotating by $2\pi$ (a full turn) gives $q \to -q$, not back to $q$.  A rotation by $4\pi$ is needed to return to the start.

This is the **double cover** of SO(3) by SU(2) — and it explains why a spin-$\frac{1}{2}$ particle must rotate by $720°$ to return to its original state.


In [ ]:
# Visualise: rotate |0> around Y axis for a full 360°, then another 360°
def bloch_ry(theta):
    return np.array([np.sin(theta), 0, np.cos(theta)])

thetas = np.linspace(0, 4 * np.pi, 200)
path = np.array([bloch_ry(t) for t in thetas])

# Midpoint marker (after 360°)
mid = len(path) // 2

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection="3d")

# Draw the full path in two colours
from helpers.plotting import draw_bloch_sphere
draw_bloch_sphere(ax=ax, vectors=[
    (path[0], "start (0°)", "green"),
    (path[mid], "360°", "orange"),
], title="Ry — 360° then another 360°")
ax.plot(path[:mid, 0], path[:mid, 1], path[:mid, 2], "-", color="cornflowerblue", lw=2, label="0→360°")
ax.plot(path[mid:, 0], path[mid:, 1], path[mid:, 2], "-", color="salmon", lw=2, label="360°→720°")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()


## Summary

| Concept | Key point |
|---|---|
| **Structure** | $q = w + x\mathbf{i} + y\mathbf{j} + z\mathbf{k}$: scalar + vector |
| **Unit norm** | $|q|=1$ required for a valid rotation |
| **Conjugate** | $q^* = w - \vec{v}$; equals inverse for unit quaternions |
| **Axis-angle** | $w = \cos(\theta/2)$, $\vec{v} = \sin(\theta/2)\hat{n}$ |
| **Double cover** | $q$ and $-q$ are the same rotation; $4\pi$ to return to start |

In `rqm-core`, `Quaternion.from_axis_angle(axis, theta)` handles all of this for you.

**Next:** [02_spinor_to_bloch.ipynb](02_spinor_to_bloch.ipynb) — connecting quantum state vectors to the Bloch sphere.
